# 01 · Preço da soja

Preço mensal do FMI (US$/t, FRED `PSOYBUSDM`), alvo e features de preço.

In [ ]:
import sys
from pathlib import Path

# No Kaggle o projeto está dentro de /kaggle/input; no computador, é a pasta acima de notebooks/
if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, dados, features, avaliacao

ETAPA = "01_preco"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})
print("Projeto em", RAIZ, "| resultados em", PASTA)

## 1. Preço

In [ ]:
preco_bruto = dados.baixar_preco()
preco = preco_bruto.loc[config.INICIO:]
print(f"{len(preco)} meses, de {preco.index[0]} a {preco.index[-1]}")

preco.plot(color="tab:green", title="Preço da soja (US$/t)")
plt.axvspan(pd.Period(config.TESTE_INICIO), pd.Period(config.TESTE_FIM), color="gray", alpha=.15, label="período de teste")
plt.legend(); plt.show()
preco.tail()

## 2. Variação mensal

Prevejo a variação log, `ln P(t) − ln P(t−1)`, e não o preço, que muda muito de patamar ao longo dos anos.

In [ ]:
variacao = np.log(preco).diff()

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5), gridspec_kw={"width_ratios": [3, 1]})
cores = np.where(variacao >= 0, "tab:green", "tab:red")
ax[0].bar(variacao.index.to_timestamp(), 100 * variacao, width=25, color=cores)
ax[0].set_title("Variação do preço no mês (%)")
ax[1].hist(100 * variacao.dropna(), bins=40, color="tab:blue")
ax[1].set_title("Distribuição da variação (%)")
fig.tight_layout(); fig.savefig(PASTA / "variacao.png", dpi=120); plt.show()

print(f"Variação média: {100 * variacao.mean():+.2f}% | desvio padrão: {100 * variacao.std():.2f}%")
print("Maiores altas e quedas:")
pd.concat([variacao.nlargest(3), variacao.nsmallest(3)]).mul(100).round(1)

## 3. Alvo e features de preço

- `alvo`: variação do mês seguinte (vazio na última linha)
- `dlog_preco`: variação do mês
- `dlog_preco_lag1`: variação do mês anterior
- `tendencia_3m`: variação nos últimos 3 meses

In [ ]:
tabela = pd.concat([preco, features.features_preco(preco), features.alvo(preco)], axis=1)

# conferência: o alvo de um mês tem que ser igual à variação do mês seguinte
assert np.allclose(tabela["alvo"].iloc[:-1].values, tabela["dlog_preco"].iloc[1:].values)

# correlação de cada feature com o alvo, olhando só os anos antes do teste
treino = tabela.loc[:pd.Period(config.TESTE_INICIO) - 1]
corr = treino.drop(columns=["preco_usd_t", "alvo"]).corrwith(treino["alvo"])
corr.plot.bar(color="tab:blue", rot=0, title=f"Correlação com o alvo ({treino.index[0]} a {treino.index[-1]})")
plt.axhline(0, color="k", lw=.8); plt.show()

tabela.tail()

## 4. Passeio aleatório

Prevê variação zero. É a referência que todo modelo precisa bater.

In [ ]:
teste = tabela.loc[config.TESTE_INICIO:config.TESTE_FIM, "alvo"]
rmse_passeio = avaliacao.rmse(teste)
print(f"Teste: {teste.index[0]} a {teste.index[-1]} ({len(teste)} meses)")
print(f"RMSE do passeio aleatório: {rmse_passeio:.4f} (cerca de {100 * rmse_passeio:.1f}% por mês)")

por_ano = teste.groupby(teste.index.year).apply(avaliacao.rmse)
por_ano.mul(100).plot.bar(color="tab:gray", rot=0, title="Erro do passeio aleatório por ano (RMSE, %)")
plt.axhline(100 * rmse_passeio, color="k", ls="--", lw=1, label="todo o teste"); plt.legend(); plt.show()

## 5. Salvando

In [ ]:
tabela.index = tabela.index.astype(str)
tabela.index.name = "mes"
tabela.to_csv(PASTA / "preco.csv")
config.salvar_config(
    PASTA, fonte="FRED PSOYBUSDM (FMI, US$/t)", meses=len(tabela),
    ultimo_mes=tabela.index[-1], rmse_passeio_teste=round(rmse_passeio, 6),
)
print(sorted(p.name for p in PASTA.iterdir()))